# Validación con tráfico NetFlow real

Reentrena el autoencoder denso y el denoising autoencoder con masking mediante Dropout sobre tráfico NetFlow benigno y los evalúa sobre una captura que contiene un ataque de inundación SYN con escaneo de puertos. Las decisiones de cada detector se explican con un EBM sustituto.

Los datos no son públicos, por lo que el cuaderno solo puede ejecutarse en el servidor donde se alojan. La variable de entorno `NETFLOW_ROOT` debe apuntar a la raíz del proyecto en ese servidor.

In [1]:
import os, sys, random, time
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.stats import spearmanr
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from interpret.glassbox import ExplainableBoostingClassifier

os.chdir(os.environ.get("NETFLOW_ROOT", os.getcwd()))
sys.path.insert(0, "data/netflowds")
import netflowds
import netflowds_atk
from netflowds import schema

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
# Tráfico benigno con muestreo sistemático
FRAC = 0.01
MAX_SAMPLES = 300_000

ds_benigno = netflowds.NetFlowWindowedDataset(window_size=1, stride=1, subsample_frags=FRAC)
n = len(ds_benigno)
print(f"Dataset benigno: {n} filas disponibles")

step = max(1, n // MAX_SAMPLES)
indices = list(range(0, n, step))[:MAX_SAMPLES]

X_raw, mask_raw = [], []
for idx in indices:
    x, mask, meta = ds_benigno[idx]
    X_raw.append(x)
    mask_raw.append(mask)

X_raw = np.array(X_raw).squeeze(axis=1)
mask_raw = np.array(mask_raw).squeeze(axis=1)
print(f"Shape final: {X_raw.shape}")

Dataset benigno: 6804929 filas disponibles
Shape final: (300000, 79)


In [4]:
feature_names = schema.FEATURE_COLUMNS

# Se descartan las columnas con muchos nulos, de fecha, constantes, de identidad y de jerarquía de cliente
IP_COLS = {f"{prefix}_ip__{i}" for prefix in ["src", "dst", "device", "lan", "wan"] for i in range(4)}
IDENTITY_COLS = {
    "device_id", "serial_number", "engine_id", "customer_id",
    "observation_id", "flow_exporter",
    "in_interface_name", "out_interface_name",
    "lan_interface_name", "wan_interface_name",
}
LEVEL_COLS = {f"level__{i}" for i in range(2, 7)}  # identifica al cliente y a la sede
# Sin LEVEL_COLS en DROP_COLS se obtienen las 37 características de la primera versión (Tabla 13)

DROP_COLS = set(schema.HIGH_NULL_COLS) | {
    "timestamp__year", "timestamp__month", "timestamp__day",
    "timestamp__second", "type", "level__0", "level__1",
} | IP_COLS | IDENTITY_COLS | LEVEL_COLS

cols_to_keep = np.array([i for i, name in enumerate(feature_names) if name not in DROP_COLS])
kept_names = [feature_names[i] for i in cols_to_keep]

print(f"Columnas descartadas: {len(DROP_COLS)}")
print(f"Columnas mantenidas: {len(cols_to_keep)} de {len(feature_names)}")

Columnas descartadas: 47
Columnas mantenidas: 32 de 79


In [12]:
# Filas sin valores ausentes y escalado
X_clean = X_raw[:, cols_to_keep]
mask_clean = mask_raw[:, cols_to_keep]

valid_rows = mask_clean.all(axis=1)
X_final = X_clean[valid_rows]
print(f"Filas utilizables: {X_final.shape[0]}")

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_final)

Filas utilizables: 264723


In [7]:
# Autoencoder denso
input_dim = X_scaled.shape[1]

tf.keras.utils.set_random_seed(SEED)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(input_dim,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(input_dim, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='mse')

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(X_scaled, X_scaled, epochs=100, validation_split=0.1,
                    callbacks=[early_stop], verbose=0)

In [ ]:
# Umbral del AE, percentil 99 del error de entrenamiento
err_train = np.mean(np.square(X_scaled - model.predict(X_scaled, verbose=0)), axis=1)
threshold = np.percentile(err_train, 99)
print(f"Umbral AE: {threshold:.5f}")

Umbral AE: 0.00862


In [5]:
# Índices de ataque y benignos, sin cargar todavía las características
ds_atk = netflowds_atk.NetFlowWindowedDataset(window_size=1, stride=1, cache_path="atk_index_1_1.npz")

t0 = time.time()
y = ds_atk.window_labels()
print(f"Etiquetas en {time.time()-t0:.1f}s, total {len(y)} flujos")

idx_ataque = np.nonzero(y)[0]
idx_benigno = np.nonzero(y == 0)[0]
print(f"Ataque: {len(idx_ataque)}, benigno: {len(idx_benigno)}")

Etiquetas en 6.0s, total 1418111556 flujos
Ataque: 627033, benigno: 1417484523


In [7]:
# Muestreo sistemático de la captura de ataque (unos 15-20 min)
N_ATTACK = 10_000
N_BENIGN = 10_000

sel_atk = idx_ataque[::max(1, len(idx_ataque) // N_ATTACK)][:N_ATTACK]
sel_ben = idx_benigno[::max(1, len(idx_benigno) // N_BENIGN)][:N_BENIGN]
sel_total = np.concatenate([sel_atk, sel_ben])

X_atk_raw, mask_atk_raw, labels_atk = [], [], []
for idx in sel_total:
    x, mask, meta = ds_atk[int(idx)]
    X_atk_raw.append(x)
    mask_atk_raw.append(mask)
    labels_atk.append(meta["label"])

X_atk_raw = np.array(X_atk_raw).squeeze(axis=1)
mask_atk_raw = np.array(mask_atk_raw).squeeze(axis=1)
labels_atk = np.array(labels_atk)
print(f"Shape: {X_atk_raw.shape}, ataques en la muestra: {labels_atk.sum()}")

Shape: (20000, 79), ataques en la muestra: 10000


In [ ]:
# Preprocesado de la muestra de ataque y evaluación del AE
X_atk_clean = X_atk_raw[:, cols_to_keep]
mask_atk_clean = mask_atk_raw[:, cols_to_keep]

valid_atk = mask_atk_clean.all(axis=1)
X_atk_final = X_atk_clean[valid_atk]
labels_final = labels_atk[valid_atk]
print(f"Filas válidas: {X_atk_final.shape[0]} de {X_atk_raw.shape[0]}")

X_atk_scaled = scaler.transform(X_atk_final)  # el escalador se ajustó solo con tráfico benigno

def metricas(nombre, err, thr):
    pred = err > thr
    tp = np.sum(pred & (labels_final == 1)); fn = np.sum(~pred & (labels_final == 1))
    fp = np.sum(pred & (labels_final == 0)); tn = np.sum(~pred & (labels_final == 0))
    print(f"{nombre}: AUC={roc_auc_score(labels_final, err):.4f}  TP={tp} FN={fn} FP={fp} TN={tn}  "
          f"recall={tp/(tp+fn):.3f}  FPR={fp/(fp+tn):.3f}  precisión={tp/(tp+fp):.3f}")
    return pred

err_atk = np.mean(np.square(X_atk_scaled - model.predict(X_atk_scaled, verbose=0)), axis=1)
pred_final = metricas("AE ", err_atk, threshold)

Filas válidas: 18831 de 20000


AE : AUC=0.8345  TP=1152 FN=8848 FP=441 TN=8390  recall=0.115  FPR=0.050  precisión=0.723


In [ ]:
# Error por columna del AE en los TP y en los FP
err_per_feature = np.square(X_atk_scaled - model.predict(X_atk_scaled, verbose=0))

tp_mask = pred_final & (labels_final == 1)
fp_mask = pred_final & (labels_final == 0)
contrib_tp = err_per_feature[tp_mask].mean(axis=0)
contrib_fp = err_per_feature[fp_mask].mean(axis=0)

print("Top features en ataques detectados (TP):")
for i in np.argsort(contrib_tp)[::-1][:10]:
    print(f"  {kept_names[i]}: {contrib_tp[i]:.5f}")

print("\nTop features en falsos positivos (FP):")
for i in np.argsort(contrib_fp)[::-1][:10]:
    print(f"  {kept_names[i]}: {contrib_fp[i]:.5f}")

Top features en ataques detectados (TP):
  sum_pkts: 0.12300
  sum_bytes: 0.06509
  wan_l4_port: 0.05164
  src_port: 0.04517
  tcp_flags: 0.02156
  selector_name: 0.01094
  saas_name: 0.00551
  events: 0.00448
  dst_port: 0.00249
  host_category: 0.00192

Top features en falsos positivos (FP):
  wan_l4_port: 0.07388
  tos: 0.06458
  events: 0.06094
  dst_port: 0.04388
  src_port: 0.04139
  application_id__id: 0.03511
  sum_bytes: 0.03134
  sum_pkts: 0.03102
  host_category: 0.02984
  saas_name: 0.02601


In [ ]:
# Denoising autoencoder con masking mediante Dropout
MASK_RATE = 0.1  # fijado de antemano, no se ajusta sobre este conjunto

tf.keras.utils.set_random_seed(SEED)
dae_in = tf.keras.Input(shape=(input_dim,))
h = tf.keras.layers.Dropout(MASK_RATE)(dae_in)  # masking con reescalado, solo activo al entrenar
for units in [64, 32, 16, 32, 64]:
    h = tf.keras.layers.Dense(units, activation='relu')(h)
dae_out = tf.keras.layers.Dense(input_dim, activation='sigmoid')(h)
dae = tf.keras.Model(dae_in, dae_out)
dae.compile(optimizer='adam', loss='mse')

history_dae = dae.fit(X_scaled, X_scaled, epochs=100, validation_split=0.1,
                      callbacks=[early_stop], verbose=0)

In [ ]:
# Umbral del DAE y comparación con el AE
err_train_dae = np.mean(np.square(X_scaled - dae.predict(X_scaled, verbose=0)), axis=1)
threshold_dae = np.percentile(err_train_dae, 99)
print(f"Umbral DAE: {threshold_dae:.5f}\n")

err_atk_dae = np.mean(np.square(X_atk_scaled - dae.predict(X_atk_scaled, verbose=0)), axis=1)
pred_final = metricas("AE ", err_atk, threshold)
pred_dae = metricas("DAE", err_atk_dae, threshold_dae)

Umbral DAE: 0.01405

AE : AUC=0.8345  TP=1152 FN=8848 FP=441 TN=8390  recall=0.115  FPR=0.050  precisión=0.723
DAE: AUC=0.7461  TP=245 FN=9755 FP=351 TN=8480  recall=0.025  FPR=0.040  precisión=0.411


In [ ]:
# EBM sustituto que imita las decisiones de cada detector, entrenado con los valores sin escalar
X_ebm = pd.DataFrame(X_atk_final.astype(float), columns=kept_names)

def ebm_surrogate(nombre, pred, err_cols):
    # La etiqueta del EBM es la decisión del detector, no la etiqueta real del flujo
    Xtr, Xte, ytr, yte = train_test_split(X_ebm, pred.astype(int), test_size=0.3,
                                          random_state=SEED, stratify=pred)
    ebm = ExplainableBoostingClassifier(random_state=SEED)
    ebm.fit(Xtr, ytr)
    p = ebm.predict_proba(Xte)[:, 1]
    print(f"[{nombre}] fidelidad al detector: accuracy={accuracy_score(yte, p > 0.5):.3f}, "
          f"AUC={roc_auc_score(yte, p):.3f} (clase mayoritaria: {max(yte.mean(), 1 - yte.mean()):.3f})")

    imp = pd.Series(ebm.term_importances(), index=ebm.term_names_).sort_values(ascending=False)
    print(f"\n[{nombre}] importancia global EBM (top 15):")
    print(imp.head(15).to_string())

    contrib = pd.DataFrame(np.abs(ebm.eval_terms(X_ebm)), columns=ebm.term_names_)
    tp = pred & (labels_final == 1)
    fp = pred & (labels_final == 0)
    print(f"\n[{nombre}] contribución media |EBM| en TP y FP (top 10):")
    print(pd.DataFrame({"TP": contrib[tp].mean(), "FP": contrib[fp].mean()})
          .sort_values("TP", ascending=False).head(10).to_string())

    rho = spearmanr(imp.reindex(kept_names).values, err_cols[pred].mean(axis=0)).correlation
    print(f"\n[{nombre}] Spearman importancia EBM vs error por columna (flujos marcados): {rho:.3f}")
    return ebm

In [ ]:
ebm_ae = ebm_surrogate("AE", pred_final, err_per_feature)

[AE] fidelidad al detector: accuracy=0.984, AUC=0.992 (clase mayoritaria: 0.915)

[AE] importancia global EBM (top 15):
wan_l4_port               1.367762
dst_port                  0.933569
src_port                  0.884812
events                    0.604264
sum_bytes                 0.557817
lan_l4_port               0.532951
sum_pkts                  0.505741
saas_app__id              0.497603
application_name__id      0.452440
lan_ip_as_name            0.449463
saas_family               0.439702
selector_name             0.416838
tcp_flags                 0.403180
dst_port & lan_l4_port    0.371936
src_port & lan_l4_port    0.367160

[AE] contribución media |EBM| en TP y FP (top 10):
                               TP        FP
wan_l4_port              3.010551  0.665902
dst_port                 2.136491  0.615263
sum_pkts                 1.810560  0.808549
wan_l4_port & sum_pkts   1.065994  0.266291
src_port                 1.043470  0.602648
tcp_flags                0.881201  0.18

In [ ]:
err_per_feature_dae = np.square(X_atk_scaled - dae.predict(X_atk_scaled, verbose=0))
ebm_dae = ebm_surrogate("DAE", pred_dae, err_per_feature_dae)

[DAE] fidelidad al detector: accuracy=0.987, AUC=0.979 (clase mayoritaria: 0.968)

[DAE] importancia global EBM (top 15):
selector_name                 0.713963
events                        0.490701
wan_l4_port                   0.476317
lan_l4_port                   0.462071
sum_pkts                      0.434398
sum_bytes                     0.407551
src_port                      0.369682
wan_ip_as_name                0.345917
dst_port                      0.315856
tos                           0.279543
wan_l4_port & saas_app__id    0.257471
wan_l4_port & sum_bytes       0.252165
lan_ip_country_code           0.249176
saas_family                   0.248970
saas_name                     0.246988

[DAE] contribución media |EBM| en TP y FP (top 10):
                                   TP        FP
events                       3.036418  1.104113
wan_l4_port                  1.480762  0.372553
wan_l4_port & events         1.227549  0.141461
src_port & dst_port          0.714440  0.217872


In [ ]:
# Valores más frecuentes de las variables con más peso en el EBM del DAE
imp_dae = pd.Series(ebm_dae.term_importances(), index=ebm_dae.term_names_)
top_cols = [t for t in imp_dae.sort_values(ascending=False).index if " & " not in t][:5]

grupos = {
    "train": X_final,
    "TN": X_atk_final[~pred_final & (labels_final == 0)],
    "FP": X_atk_final[fp_mask],
    "TP": X_atk_final[tp_mask],
}
for col in top_cols:
    j = kept_names.index(col)
    print(f"\n{col}")
    for g, X in grupos.items():
        v, c = np.unique(X[:, j], return_counts=True)
        top = sorted(zip(c, v), reverse=True)[:5]
        print(f"  {g:5s} n_valores={len(v):6d}  top: " + ", ".join(f"{int(val)}({cnt/len(X):.0%})" for cnt, val in top))


selector_name
  train n_valores=    54  top: 72(10%), 4(10%), 122(9%), 79(8%), 43(8%)
  TN    n_valores=    89  top: 122(15%), 105(10%), 56(8%), 58(7%), 4(6%)
  FP    n_valores=    50  top: 56(17%), 105(9%), 43(9%), 79(9%), 76(7%)
  TP    n_valores=     1  top: 56(100%)

events
  train n_valores=     5  top: 1233(89%), 2247(10%), 2966(1%), 3524(0%), 3979(0%)
  TN    n_valores=     4  top: 1233(92%), 2247(7%), 2966(1%), 3524(0%)
  FP    n_valores=     4  top: 1233(66%), 2247(28%), 2966(6%), 3524(0%)
  TP    n_valores=     3  top: 1233(95%), 2247(4%), 2966(1%)

wan_l4_port
  train n_valores= 15817  top: 443(25%), 53(22%), 179(5%), 60000(5%), 0(4%)
  TN    n_valores=   718  top: 443(39%), 53(17%), 8000(7%), 80(4%), 0(3%)
  FP    n_valores=    96  top: 443(44%), 47808(9%), 53(6%), 80(6%), 3128(3%)
  TP    n_valores=  1135  top: 35138(0%), 34833(0%), 32571(0%), 31578(0%), 29211(0%)

lan_l4_port
  train n_valores= 34165  top: 161(6%), 0(3%), 2048(2%), 8001(1%), 7680(1%)
  TN    n_valores=  

In [ ]:
# Ataques detectados (TP) frente a no detectados (FN) por el AE
fn_mask = ~pred_final & (labels_final == 1)
grupos = {"TP": X_atk_final[tp_mask], "FN": X_atk_final[fn_mask]}

for col in ["lan_l4_port", "sum_pkts", "tcp_flags", "events"]:
    j = kept_names.index(col)
    print(f"\n{col}")
    for g, X in grupos.items():
        v, c = np.unique(X[:, j], return_counts=True)
        top = sorted(zip(c, v), reverse=True)[:5]
        print(f"  {g:3s} n_valores={len(v):6d}  top: " + ", ".join(f"{int(val)}({cnt/len(X):.0%})" for cnt, val in top))


lan_l4_port
  TP  n_valores=     3  top: 23(91%), 22(5%), 443(4%)
  FN  n_valores=    30  top: 23(45%), 22(40%), 443(15%), 65342(0%), 63847(0%)

sum_pkts
  TP  n_valores=   156  top: 2330(4%), 2486(2%), 2664(2%), 2473(2%), 2683(2%)
  FN  n_valores=   204  top: -759(17%), -465(12%), 2304(6%), -256(6%), 2325(4%)

tcp_flags
  TP  n_valores=     3  top: 181(91%), 63(5%), 142(4%)
  FN  n_valores=     5  top: 125(48%), 63(26%), 181(19%), 142(7%), 132(0%)

events
  TP  n_valores=     3  top: 1233(95%), 2247(4%), 2966(1%)
  FN  n_valores=     3  top: 1233(89%), 2247(10%), 2966(1%)


In [ ]:
# Funciones de forma del EBM del AE para la figura de figuras/figuras_cap5.py
EBM = ebm_ae
OUT = "figuras_ebm"
os.makedirs(OUT, exist_ok=True)

pd.DataFrame({"termino": EBM.term_names_,
              "importancia": EBM.term_importances()}).to_csv(f"{OUT}/importancia_global.csv", index=False)

g = EBM.explain_global()
for nombre in ["dst_port", "wan_l4_port", "sum_pkts", "tcp_flags"]:
    d = g.data(EBM.term_names_.index(nombre))
    pd.DataFrame({"limite": d["names"][:len(d["scores"])],
                  "score": d["scores"]}).to_csv(f"{OUT}/forma_{nombre}.csv", index=False)

print(os.listdir(OUT))

['forma_sum_pkts.csv', 'importancia_global.csv', 'forma_tcp_flags.csv', 'forma_dst_port.csv', 'forma_wan_l4_port.csv']


In [17]:
# AE y DAE con las semillas 42, 1, 2, 3 y 4
from scipy.stats import ttest_rel, wilcoxon

SEEDS = [42, 1, 2, 3, 4]  # las mismas que en los conjuntos de referencia
PCTS = [90, 95, 97, 99]

# Mismo preprocesado de la muestra de ataque que en la evaluación del AE
X_atk_clean = X_atk_raw[:, cols_to_keep]
valid_atk = mask_atk_raw[:, cols_to_keep].all(axis=1)
X_atk_final = X_atk_clean[valid_atk]
labels_final = labels_atk[valid_atk]
X_atk_scaled = scaler.transform(X_atk_final)
input_dim = X_scaled.shape[1]
print(f"Train {X_scaled.shape}, test {X_atk_scaled.shape}, ataques {labels_final.sum()}")

def construir(tipo):
    inp = tf.keras.Input(shape=(input_dim,))
    h = tf.keras.layers.Dropout(MASK_RATE)(inp) if tipo == "DAE" else inp
    for units in [64, 32, 16, 32, 64]:
        h = tf.keras.layers.Dense(units, activation='relu')(h)
    out = tf.keras.layers.Dense(input_dim, activation='sigmoid')(h)
    m = tf.keras.Model(inp, out)
    m.compile(optimizer='adam', loss='mse')
    return m

filas = []

for s in SEEDS:
    for tipo in ["AE", "DAE"]:
        t0 = time.time()
        tf.keras.utils.set_random_seed(s)
        m = construir(tipo)
        es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
        h = m.fit(X_scaled, X_scaled, epochs=100, validation_split=0.1, callbacks=[es], verbose=0)
        err_tr = np.mean(np.square(X_scaled - m.predict(X_scaled, verbose=0)), axis=1)
        err_te = np.mean(np.square(X_atk_scaled - m.predict(X_atk_scaled, verbose=0)), axis=1)
        fila = {"seed": s, "modelo": tipo, "epocas": len(h.history['loss']),
                "AUC": roc_auc_score(labels_final, err_te)}
        for p in PCTS:
            pred = err_te > np.percentile(err_tr, p)
            tp = np.sum(pred & (labels_final == 1)); fp = np.sum(pred & (labels_final == 0))
            fila[f"rec_p{p}"] = tp / labels_final.sum()
            fila[f"fpr_p{p}"] = fp / (labels_final == 0).sum()
            fila[f"prec_p{p}"] = tp / max(tp + fp, 1)
        filas.append(fila)
        print(f"seed={s:2d} {tipo:3s} épocas={fila['epocas']:3d} AUC={fila['AUC']:.4f} "
              f"rec_p99={fila['rec_p99']:.3f} fpr_p99={fila['fpr_p99']:.3f} ({time.time()-t0:.0f}s)")

res = pd.DataFrame(filas)

print("\nMedia ± desviación (5 semillas)")
cols = ["AUC"] + [f"{k}_p{p}" for p in PCTS for k in ["rec", "fpr", "prec"]]
resumen = res.groupby("modelo")[cols].agg(["mean", "std"])
for mod in ["AE", "DAE"]:
    print(f"\n{mod}")
    for c in cols:
        print(f"  {c:9s} {resumen.loc[mod, (c, 'mean')]:.4f} ± {resumen.loc[mod, (c, 'std')]:.4f}")

ae = res[res.modelo == "AE"].set_index("seed").loc[SEEDS]
dae = res[res.modelo == "DAE"].set_index("seed").loc[SEEDS]
print("\nAE vs DAE pareado por semilla")
for c in ["AUC", "rec_p99", "rec_p90"]:
    d = ae[c] - dae[c]
    print(f"  {c:8s} AE−DAE={d.mean():+.4f}  AE gana en {(d > 0).sum()}/5  "
          f"t-test p={ttest_rel(ae[c], dae[c]).pvalue:.3f}  Wilcoxon p={wilcoxon(ae[c], dae[c]).pvalue:.3f}")

Train (264723, 32), test (18831, 32), ataques 10000


seed=42 AE  épocas=100 AUC=0.8345 rec_p99=0.115 fpr_p99=0.050 (343s)


seed=42 DAE épocas= 32 AUC=0.7461 rec_p99=0.025 fpr_p99=0.040 (119s)


seed= 1 AE  épocas=100 AUC=0.8290 rec_p99=0.053 fpr_p99=0.069 (346s)


seed= 1 DAE épocas= 30 AUC=0.7767 rec_p99=0.000 fpr_p99=0.052 (111s)


...

seed= 2 DAE épocas= 56 AUC=0.7378 rec_p99=0.000 fpr_p99=0.043 (201s)


seed= 3 AE  épocas=100 AUC=0.9128 rec_p99=0.373 fpr_p99=0.037 (343s)


seed= 3 DAE épocas= 42 AUC=0.8107 rec_p99=0.098 fpr_p99=0.036 (153s)


seed= 4 AE  épocas=100 AUC=0.8090 rec_p99=0.000 fpr_p99=0.036 (346s)


seed= 4 DAE épocas= 73 AUC=0.7815 rec_p99=0.000 fpr_p99=0.037 (263s)

Media ± desviación (5 semillas)

AE
  AUC       0.8558 ± 0.0449
  rec_p90   0.8839 ± 0.1012
  fpr_p90   0.2673 ± 0.0160
  prec_p90  0.7879 ± 0.0211
  rec_p95   0.6052 ± 0.1809
  fpr_p95   0.1548 ± 0.0182
  prec_p95  0.8084 ± 0.0397
  rec_p97   0.3955 ± 0.2643
  fpr_p97   0.1016 ± 0.0237
  prec_p97  0.7564 ± 0.1379
  rec_p99   0.1291 ± 0.1440
  fpr_p99   0.0460 ± 0.0142
  prec_p99  0.5731 ± 0.3579

DAE
  AUC       0.7706 ± 0.0293
  rec_p90   0.5137 ± 0.0584
  fpr_p90   0.2229 ± 0.0175
  prec_p90  0.7219 ± 0.0253
  rec_p95   0.2046 ± 0.0692
  fpr_p95   0.1416 ± 0.0131
  prec_p95  0.6088 ± 0.0809
  rec_p97   0.0673 ± 0.0665
  fpr_p97   0.1001 ± 0.0115
  prec_p97  0.3512 ± 0.2539
  rec_p99   0.0244 ± 0.0422
  fpr_p99   0.0416 ± 0.0064
  prec_p99  0.2331 ± 0.3400

AE vs DAE pareado por semilla
  AUC      AE−DAE=+0.0853  AE gana en 5/5  t-test p=0.018  Wilcoxon p=0.062
  rec_p99  AE−DAE=+0.1047  AE gana en 5/5  t-test p=0.

In [8]:
# Caso de 37 características (sin descartar la jerarquía de cliente level__2 a level__6), semilla 42 (Tabla 13)
DROP_COLS_37 = DROP_COLS - LEVEL_COLS
cols_37 = np.array([i for i, name in enumerate(feature_names) if name not in DROP_COLS_37])
print(f"Columnas mantenidas: {len(cols_37)} de {len(feature_names)}")

X_37 = X_raw[:, cols_37][mask_raw[:, cols_37].all(axis=1)]
scaler_37 = MinMaxScaler()
X_37_scaled = scaler_37.fit_transform(X_37)

valid_37 = mask_atk_raw[:, cols_37].all(axis=1)
X_atk_37 = scaler_37.transform(X_atk_raw[:, cols_37][valid_37])
y_37 = labels_atk[valid_37]
dim_37 = X_37_scaled.shape[1]
print(f"Entrenamiento {X_37_scaled.shape}, evaluación {X_atk_37.shape}, ataques {y_37.sum()}")


def evaluar_37(nombre, m):
    m.fit(X_37_scaled, X_37_scaled, epochs=100, validation_split=0.1, verbose=0,
          callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)])
    thr = np.percentile(np.mean(np.square(X_37_scaled - m.predict(X_37_scaled, verbose=0)), axis=1), 99)
    err = np.mean(np.square(X_atk_37 - m.predict(X_atk_37, verbose=0)), axis=1)
    pred = err > thr
    tp = np.sum(pred & (y_37 == 1)); fn = np.sum(~pred & (y_37 == 1))
    fp = np.sum(pred & (y_37 == 0)); tn = np.sum(~pred & (y_37 == 0))
    print(f"{nombre}: AUC={roc_auc_score(y_37, err):.4f}  TP={tp} FN={fn} FP={fp} TN={tn}  "
          f"recall={tp/(tp+fn):.3f}  FPR={fp/(fp+tn):.3f}")


# Autoencoder denso, con la misma construcción que el de arriba
tf.keras.utils.set_random_seed(SEED)
ae_37 = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(dim_37,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(dim_37, activation='sigmoid'),
])
ae_37.compile(optimizer='adam', loss='mse')
evaluar_37("AE ", ae_37)

# Denoising autoencoder con masking mediante Dropout al 10 %, con la misma construcción que el de arriba
tf.keras.utils.set_random_seed(SEED)
entrada = tf.keras.Input(shape=(dim_37,))
h = tf.keras.layers.Dropout(0.1)(entrada)
for units in [64, 32, 16, 32, 64]:
    h = tf.keras.layers.Dense(units, activation='relu')(h)
dae_37 = tf.keras.Model(entrada, tf.keras.layers.Dense(dim_37, activation='sigmoid')(h))
dae_37.compile(optimizer='adam', loss='mse')
evaluar_37("DAE", dae_37)

Columnas mantenidas: 37 de 79
Entrenamiento (264723, 37), evaluación (18831, 37), ataques 10000


AE : AUC=0.9114  TP=10000 FN=0 FP=1451 TN=7380  recall=1.000  FPR=0.164


DAE: AUC=0.9371  TP=10000 FN=0 FP=1272 TN=7559  recall=1.000  FPR=0.144
